In [1]:
from pathlib import Path
import tempfile

from embedding_lab.chunking import MarkdownChunker
from embedding_lab.documents import collect_markdown_paths, load_documents

In [2]:
knowledge_dir = Path(tempfile.mkdtemp(prefix="studyagent-chunks-"))

(knowledge_dir / "README.md").write_text(
    "# 目录说明\n\n这里只负责导航。",
    encoding="utf-8",
)
(knowledge_dir / "walls.md").write_text(
    """---
status: approved
---
# 墙体

## 幕墙

幕墙需要记录父墙体关系。

## 承重墙

承重墙负责传递上部荷载。
""",
    encoding="utf-8",
)
(knowledge_dir / "BLUEPRINT-SPEC-MINIMAL.md").write_text(
    "# 不应进入普通 RAG",
    encoding="utf-8",
)

print("临时知识库：", knowledge_dir)

临时知识库： C:\Users\ADMINI~1\AppData\Local\Temp\studyagent-chunks-1y_aezy_


In [3]:
paths = collect_markdown_paths(knowledge_dir)
documents = load_documents(knowledge_dir)

print("扫描路径：", [path.name for path in paths])
assert "BLUEPRINT-SPEC-MINIMAL.md" not in [path.name for path in paths]
assert all("status: approved" not in document.text for document in documents)
assert any(document.doc_scope == "index" for document in documents)

扫描路径： ['README.md', 'walls.md']


In [4]:
chunker = MarkdownChunker(chunk_size=200, chunk_overlap=20)
chunks = chunker.split_documents(documents, namespace="chunking_test")

for chunk in chunks:
    print("\n---")
    print("id:", chunk.id[:16])
    print("source:", chunk.metadata["source"])
    print("scope:", chunk.metadata["doc_scope"])
    print("heading:", chunk.metadata["heading"])
    print(chunk.text)

assert any("墙体 > 幕墙" in chunk.metadata["heading"] for chunk in chunks)

second_run = chunker.split_documents(documents, namespace="chunking_test")
assert [chunk.id for chunk in chunks] == [chunk.id for chunk in second_run]


---
id: 38cdfc3e42dd8f7c
source: README.md
scope: index
heading: 目录说明
> 知识路径：目录说明

# 目录说明

这里只负责导航。

---
id: 69e69c9708c3534f
source: walls.md
scope: generation
heading: 墙体
> 知识路径：墙体

# 墙体

---
id: 91969de5ee387b8c
source: walls.md
scope: generation
heading: 墙体 > 幕墙
> 知识路径：墙体 > 幕墙

## 幕墙

幕墙需要记录父墙体关系。

---
id: d522ef8f39dd223f
source: walls.md
scope: generation
heading: 墙体 > 承重墙
> 知识路径：墙体 > 承重墙

## 承重墙

承重墙负责传递上部荷载。
